# WorldSplat-Prune — Mip-NeRF 360 labeling

End-to-end driver for labeling the **Mip-NeRF 360** scenes (no extra Replica
scene required). **SfM / point-cloud generation is done by a separate process**,
so this notebook starts from a staged dataset and does no HLOC/COLMAP work:

```
$DATASET_ROOT/mipnerf360/<scene>/
    images/          # RGB frames
    sparse/0/        # COLMAP model, PINHOLE / SIMPLE_PINHOLE camera
```

| step | what |
|---|---|
| 4 | validate the staged dataset (images + PINHOLE sparse) |
| 5 | SAM2 -> per-frame `semantic_instance/` (instance pseudo-labels) |
| 6 | verify label/image alignment |
| 7 | minimal **3D instance fusion** -> view-consistent ids |
| 8 | optional GT reference scene (only for the metric comparison) |
| 9 | train + metrics |

Full guide and caveats: `docs/datasets.md`.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Get this repo + labeling deps

`sam2` is **labeling-only** (not needed for training). No HLOC/pycolmap here.

In [ ]:
import os

REPO_URL = "https://github.com/satyaba/SGDC-3DGS.git"
REPO_DIR = "/content/SGDC-3DGS"
BRANCH = "main"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout {BRANCH} -q
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)

!git submodule update --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn
!pip install -q plyfile opencv-python joblib
!pip install -q "git+https://github.com/facebookresearch/sam2.git"

print("cwd:", os.getcwd())

## 3. Paths + knobs

In [ ]:
from pathlib import Path

HOME_DIR  = "/content/drive/MyDrive/Magister/Thesis/Script"
DATASET_ROOT = Path(HOME_DIR) / "Dataset"
os.environ["HOME_DIR"] = HOME_DIR
os.environ["DATASET_ROOT"] = str(DATASET_ROOT)

MIP_SCENES = ["garden", "bicycle"]   # the two non-segmented scenes
STRIDE = 1        # 5 for a QUICK smoke test, 1 for the real labeling
HAS_GROUND_TRUTH = False   # True only if a scene with GT instances is placed

SCENES = {s: DATASET_ROOT / "mipnerf360" / s for s in MIP_SCENES}
print(SCENES)

## 4. Validate the staged dataset

Confirms each scene has the SfM output from the separate process and that the
camera model is **PINHOLE / SIMPLE_PINHOLE** (the rasterizer requires it).

In [ ]:
from worldsplat_prune.prep import scene_io as sio

for s, ds in SCENES.items():
    imgs = ds / "images"
    sparse = ds / "sparse" / "0"
    n_img = len([p for p in imgs.glob('*') if p.is_file()]) if imgs.is_dir() else 0
    has_model = (sparse / "cameras.bin").exists() or (sparse / "cameras.txt").exists()
    print(f"{s}: images={n_img} model={has_model} -> {ds}")
    if has_model:
        scene = sio.load_colmap_scene(sparse)
        models = sorted({c.model for c in scene.cameras})
        ok = set(models) <= {"PINHOLE", "SIMPLE_PINHOLE"}
        print(f"   {len(scene.cameras)} cameras, {scene.xyz.shape[0]} points, "
              f"cameras={models} {'OK' if ok else 'NOT 3DGS-COMPATIBLE'}")

## 5. SAM2 instance pseudo-labels

Downloads SAM2.1-large once, then writes `<dataset>/semantic_instance/<stem>.png`
(uint16 ids, `0 == void`). Existing labels are skipped (safe to re-run); set
`STRIDE = 5` above for a fast subset pass first.

**If SAM2 OOMs**, cap memory with `resize_long_side=1024` (or lower),
`points_per_side=32` (64 is ~4x the prompts), and `points_per_batch=32`.

In [ ]:
from worldsplat_prune.segment_sam2 import label_dataset

CKPT_DIR = Path(HOME_DIR) / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAM2_CKPT = CKPT_DIR / "sam2.1_hiera_large.pt"
if not SAM2_CKPT.exists():
    !wget -q --show-progress -O "{SAM2_CKPT}" https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

for s, ds in SCENES.items():
    label_dataset(
        ds, sam2_ckpt=str(SAM2_CKPT),
        sam2_cfg="configs/sam2.1/sam2.1_hiera_l.yaml",
        points_per_side=32, points_per_batch=32, resize_long_side=1024,
        min_area=100, stride=STRIDE,
    )

## 6. Verify label/image alignment

Overlays segment boundaries (red) on the RGB frames → inspect `worldsplat_align_check/<scene>/`.

In [ ]:
for s, ds in SCENES.items():
    !python -m worldsplat_prune.verify_alignment --dataset_dir "{ds}" --adapter sam2 --out "worldsplat_align_check/{s}" --n 8

## 7. Minimal 3D instance fusion → view-consistent ids

SAM2 ids restart every frame; the per-Gaussian vote needs one id per 3D
object across views. This fuses the raw `semantic_instance` maps into
`semantic_instance_fused/` (posed RGB + COLMAP only — no depth, no classifier),
plus `point_labels.npy` and `fusion_summary.json`.

Merging uses **3D point-set overlap** (`metric="jaccard"`), not shared-point
count, to avoid collapsing the scene into one blob. **Check the summary:**
`instances` should be tens–hundreds and `largest_instance_frac` should be well
below 1.0. If you see `WARNING: only one instance`, re-run SAM2 with
`--no_fill_void` (don't tile the background) and/or raise `min_score`.

In [ ]:
from worldsplat_prune.prep.fuse_instances import fuse_instances

for s, ds in SCENES.items():
    summary = fuse_instances(
        ds, labels_dir="semantic_instance", out_dir="semantic_instance_fused",
        metric="jaccard", min_score=0.25, min_views=2, color_sim=0.0,
    )
    print(s, "->", {k: summary[k] for k in ("instances", "kept_obs", "largest_instance_frac", "points_labeled")})

## 8. (Optional) GT reference scene

Only for the quantitative fusion comparison (`docs/datasets.md` §3.5). Not needed
for the Mip-NeRF 360 study. Set `HAS_GROUND_TRUTH = True` in §3 if one is placed.

In [ ]:
from worldsplat_prune.datasets import get_adapter

gt_scene = DATASET_ROOT / "office_0" / "Sequence_1"
if HAS_GROUND_TRUTH and gt_scene.exists():
    print(get_adapter("replica", gt_scene).meta())
    # python -m worldsplat_prune.prep.eval_instances --dataset_dir <scene> --gt semantic_instance_gt \
    #     --method fused=semantic_instance_fused --method raw=semantic_instance:first
else:
    print("GT-free study: skipping the metric comparison.")

## 9. Train + metrics

Same base / B / A.1 arms as `office_0`, via the manifest. `USE_FUSED = True`
trains on the fused (consistent) labels; `False` uses raw per-frame SAM2 ids.
For a quick check use the `*_sanity` runs (3k iters) first.

In [ ]:
from experiments.runs import run_by_id
from experiments.metrics import extract_run

# Train the base model on a staged Mip-NeRF 360 scene. Labels are not needed for
# training; they enable the band/interior metrics at eval time.
run_by_id("base_sanity", dataset="mipnerf360/garden", tag="garden")   # 3k smoke
extract_run("base_sanity", tag="garden",
            label_adapter="sam2", label_dir="semantic_instance_fused")

# full 30k:
# run_by_id("base_30k", dataset="mipnerf360/garden", tag="garden")
# extract_run("base_30k", tag="garden",
#             label_adapter="sam2", label_dir="semantic_instance_fused")
